# CUDA Inference Benchmark
This notebook contains the essential benchmark workflow for T4 GPU inference: baseline attention, optimized attention, paged KV cache simulation, vLLM throughput, and cost-model validation.

In [ ]:
# Set up and verify the GPU environment
!pip install -q triton vllm scikit-learn
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Baseline Attention Benchmark
Define a simple vanilla attention implementation, generate sample tensors, and measure latency, throughput, and peak CUDA memory.

In [ ]:
import torch
import time

# Baseline attention implementation
def vanilla_attention(Q, K, V):
    scores = Q @ K.transpose(-2, -1)
    probs = torch.softmax(scores, dim=-1)
    return probs @ V

# Example input geometry
batch_size = 2
num_heads = 4
seq_len = 128
head_dim = 64

Q = torch.randn(batch_size, num_heads, seq_len, head_dim, device="cuda")
K = torch.randn(batch_size, num_heads, seq_len, head_dim, device="cuda")
V = torch.randn(batch_size, num_heads, seq_len, head_dim, device="cuda")

print('Sample tensors generated on CUDA device.')

In [ ]:
_ = vanilla_attention(Q, K, V)
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
iterations = 100
start.record()
for _ in range(iterations):
    _ = vanilla_attention(Q, K, V)
end.record()
torch.cuda.synchronize()
latency_ms = start.elapsed_time(end) / iterations
throughput = (batch_size * seq_len * iterations) / (latency_ms / 1000)

torch.cuda.reset_peak_memory_stats()
_ = vanilla_attention(Q, K, V)
peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2

print(f'Baseline Latency: {latency_ms:.4f} ms')
print(f'Baseline Throughput: {throughput:.2f} elements/s')
print(f'Peak CUDA Memory: {peak_memory_mb:.2f} MB')

## Optimized Attention Comparison
Compare the naive implementation against PyTorch's `scaled_dot_product_attention`.

In [ ]:
import torch.nn.functional as F

# Optimized attention implementations
def naive_attention(Q, K, V):
    scores = (Q @ K.transpose(-2, -1)) / (Q.size(-1) ** 0.5)
    probs = torch.softmax(scores, dim=-1)
    return probs @ V

def sdpa_attention(Q, K, V):
    return F.scaled_dot_product_attention(Q, K, V)

batch, heads, seq, dim = 4, 8, 1024, 64
Q_large = torch.randn(batch, heads, seq, dim, device='cuda', dtype=torch.float16)
K_large = torch.randn(batch, heads, seq, dim, device='cuda', dtype=torch.float16)
V_large = torch.randn(batch, heads, seq, dim, device='cuda', dtype=torch.float16)

for _ in range(10):
    _ = naive_attention(Q_large, K_large, V_large)
    _ = sdpa_attention(Q_large, K_large, V_large)

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()
for _ in range(50):
    _ = naive_attention(Q_large, K_large, V_large)
end.record()
torch.cuda.synchronize()
naive_time = start.elapsed_time(end) / 50

start.record()
for _ in range(50):
    _ = sdpa_attention(Q_large, K_large, V_large)
end.record()
torch.cuda.synchronize()
sdpa_time = start.elapsed_time(end) / 50

print(f'Naive Attention: {naive_time:.4f} ms')
print(f'SDPA Attention: {sdpa_time:.4f} ms')
print(f'Speedup: {naive_time / sdpa_time:.2f}x')

## Paged KV Cache Simulation
Simulate `ContiguousKV` versus `PagedKV` to estimate memory fragmentation and savings.

In [ ]:
import numpy as np

class ContiguousKV:
    def __init__(self, max_sequences, max_len, head_dim):
        self.max_sequences = max_sequences
        self.max_len = max_len
        self.head_dim = head_dim

    def peak_memory_mb(self):
        return (self.max_sequences * self.max_len * self.head_dim * 2) / 1e6

class PagedKV:
    def __init__(self, block_size, total_blocks, head_dim):
        self.block_size = block_size
        self.head_dim = head_dim
        self.free_blocks = list(range(total_blocks))
        self.block_table = {}

    def allocate(self, seq_id, seq_len):
        num_needed = (seq_len + self.block_size - 1) // self.block_size
        allocated = [self.free_blocks.pop() for _ in range(num_needed)]
        self.block_table[seq_id] = allocated

    def used_memory_mb(self):
        used_blocks = sum(len(v) for v in self.block_table.values())
        return (used_blocks * self.block_size * self.head_dim * 2) / 1e6

    def fragmentation_pct(self, seq_id, actual_len):
        num_blocks = len(self.block_table[seq_id])
        total_capacity = num_blocks * self.block_size
        unused_slots = total_capacity - actual_len
        return unused_slots / total_capacity * 100

max_seq_len = 2048
block_size = 16
head_dim = 128
num_sequences = 10
actual_lens = np.random.randint(512, 1024, size=num_sequences)

contig = ContiguousKV(num_sequences, max_seq_len, head_dim)
paged = PagedKV(block_size=block_size, total_blocks=2000, head_dim=head_dim)
frags = []
for i, length in enumerate(actual_lens):
    paged.allocate(i, length)
    frags.append(paged.fragmentation_pct(i, length))

print(f'Contiguous peak memory: {contig.peak_memory_mb():.2f} MB')
print(f'Paged used memory: {paged.used_memory_mb():.2f} MB')
print(f'Average fragmentation: {np.mean(frags):.2f}%')

## vLLM Throughput Benchmark
Run a small vLLM workload on the GPU and report total tokens, duration, and throughput.

In [ ]:
from vllm import LLM, SamplingParams
import time

model_name = 'facebook/opt-125m'
llm = LLM(model=model_name, device='cuda', enforce_eager=True, gpu_memory_utilization=0.7)
sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=64)
prompts = [
    'The future of AI inference optimization is',
    'PagedAttention works by',
    'GPU memory bottlenecks can be solved by',
    'FlashAttention improves speed because'
]

start_time = time.time()
outputs = llm.generate(prompts, sampling_params)
duration = time.time() - start_time
tokens = sum(len(out.outputs[0].token_ids) for out in outputs)

print(f'Generated {tokens} tokens in {duration:.2f} s')
print(f'Throughput: {tokens / duration:.2f} tokens/s')
print(f'Sample output: {outputs[0].outputs[0].text}')

## Cost Model Validation
Measure attention latency across sequence lengths, fit a quadratic model, and visualize the fit.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

N_values = np.array([128, 256, 512, 1024, 2048])
latencies = []

for N in N_values:
    Q = torch.randn(4, 8, N, 64, device='cuda', dtype=torch.float16)
    K = torch.randn(4, 8, N, 64, device='cuda', dtype=torch.float16)
    V = torch.randn(4, 8, N, 64, device='cuda', dtype=torch.float16)

    for _ in range(10):
        _ = F.scaled_dot_product_attention(Q, K, V)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(30):
        _ = F.scaled_dot_product_attention(Q, K, V)
    end.record()
    torch.cuda.synchronize()

    latency = start.elapsed_time(end) / 30
    latencies.append(latency)
    print(f'N={N}: {latency:.4f} ms')

X = np.vstack([N_values**2, N_values, np.ones_like(N_values)]).T
model = LinearRegression(fit_intercept=False).fit(X, latencies)
gamma, beta, alpha = model.coef_
pred = model.predict(X)
r2 = r2_score(latencies, pred)

print(f'Fitted model: T = {gamma:.2e}N^2 + {beta:.2e}N + {alpha:.4f}')
print(f'R^2 = {r2:.4f}')

plt.plot(N_values, latencies, 'o-', label='Measured')
plt.plot(N_values, pred, '--', label='Quadratic fit')
plt.xlabel('Sequence length (N)')
plt.ylabel('Latency (ms)')
plt.title('Cost Model Validation')
plt.legend()
plt.grid(True)
plt.show()

## Final Notes
This notebook is a compact T4 benchmark workflow. It is intentionally focused on the core experiments and removes setup, cleanup, and Git interaction code.